## READ ME

* The source data that this notebook is meant to parse and dump into a csv for training originates from the repo which as of 4/18/26 resides in `data/spectra/raw/nist_IR.zip`
  * https://github.com/IvanChernyshov/NistChemData


### Purpose
* the purpose of this file is to parse the jdx files into a structured data format like a csv
  * uses the jcamp library to parse the file

### Output
* jdx_metadata.csv contains data on the compound id, compound name, a few compound properties, data on the spectroscopy equipment and techniques used, and a few descriptive statistics of the spectroscopy data
* jdx_xy_points.csv contains the spectroscopy data as a pair of cordinates (x,y) where x is the wavelength and y is the transmitance
  * logic for parsing such contained in `expand_xy_points`

In [ ]:
!pip install jcamp rdkit cirpy

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
from jcamp import jcamp_read

import cirpy
import json
import time
from rdkit import Chem
from urllib.parse import quote
from urllib.request import urlopen
from typing import Any



## Parse JDX into csv

## funcitons to parse jdx using jcamp

In [ ]:
def as_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, np.ndarray):
        return ""
    if isinstance(value, (np.integer, np.floating)):
        return str(value.item())
    return str(value).replace("\r\n", "\n").replace("\r", "\n").replace("\n", " ").strip()


def as_number(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (np.integer, np.floating)):
        return str(value.item())
    if isinstance(value, (int, float)):
        return str(value)
    return as_text(value)


def expand_xy_points(data: dict[str, Any]) -> pd.DataFrame:
    """Return explicit x/y pairs for the spectrum.

    JCAMP files often encode an IR trace as a starting x value plus a delta x,
    followed by a list of y values. For example, a line like:

        549.759 29.3 29.05 28.32 ...

    means:
        (549.759, 29.3)
        (549.759 + deltax * 1, 29.05)
        (549.759 + deltax * 2, 28.32)

    The `jcamp` reader usually gives us explicit `x` and `y` arrays already.
    If those arrays are missing, we rebuild x from `firstx` and `deltax`.
    """

    x = data.get("x")
    y = data.get("y")

    if x is not None and y is not None and len(x) == len(y) and len(x) > 0:
        return pd.DataFrame({"x": np.asarray(x, dtype=float), "y": np.asarray(y, dtype=float)})

    if y is None or len(y) == 0:
        return pd.DataFrame(columns=["x", "y"])

    y_values = np.asarray(y, dtype=float)
    firstx = data.get("firstx")
    deltax = data.get("deltax")

    if firstx is not None and deltax is not None:
        x_values = float(firstx) + float(deltax) * np.arange(len(y_values))
    else:
        x_values = np.arange(len(y_values), dtype=float)

    return pd.DataFrame({"x": x_values, "y": y_values})


def parse_jdx_file(path: Path) -> tuple[dict[str, str], pd.DataFrame]:
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        data = jcamp_read(handle)

    metadata: dict[str, str] = {"filename": path.name}
    for key, value in data.items():
        if key in {"x", "y"}:
            continue
        if isinstance(value, np.ndarray):
            continue
        metadata[key] = as_text(value)

    x = data.get("x")
    y = data.get("y")
    metadata["npoints"] = as_number(data.get("npoints", len(x) if x is not None else ""))
    metadata["x_min"] = as_number(np.min(x) if x is not None and len(x) else "")
    metadata["x_max"] = as_number(np.max(x) if x is not None and len(x) else "")
    metadata["y_min"] = as_number(np.min(y) if y is not None and len(y) else "")
    metadata["y_max"] = as_number(np.max(y) if y is not None and len(y) else "")
    metadata["firstx"] = as_number(data.get("firstx"))
    metadata["lastx"] = as_number(data.get("lastx"))
    metadata["deltax"] = as_number(data.get("deltax"))

    spectrum = expand_xy_points(data)
    if not spectrum.empty:
        spectrum.insert(0, "filename", path.name)
        spectrum.insert(1, "point_index", np.arange(len(spectrum), dtype=int))
    else:
        spectrum = pd.DataFrame(columns=["filename", "point_index", "x", "y"])

    return metadata, spectrum


### Get list of jdx files


In [ ]:
jdx_dir = Path.cwd() / "data" / "IR"
if not jdx_dir.exists():
    alt_dir = Path.cwd().parent / "data" / "IR"
    if alt_dir.exists():
        jdx_dir = alt_dir

jdx_files = sorted(jdx_dir.glob("*.jdx"))
print("jdx dir:", jdx_dir)
print("jdx files are:", len(jdx_files))
jdx_files[:5]


### parse each file and concatonate into csv

In [ ]:
metadata_rows: list[dict[str, str]] = []
xy_frames: list[pd.DataFrame] = []
failed_files: list[dict[str, str]] = []

for path in jdx_files:
    try:
        metadata, spectrum = parse_jdx_file(path)
        metadata_rows.append(metadata)
        xy_frames.append(spectrum)
    except Exception as exc:
        failed_files.append({"filename": path.name, "error": f"{type(exc).__name__}: {exc}"})

metadata_df = pd.DataFrame(metadata_rows)
xy_df = pd.concat(xy_frames, ignore_index=True) if xy_frames else pd.DataFrame(columns=["filename", "point_index", "x", "y"])
failed_df = pd.DataFrame(failed_files)

metadata_df.head()


In [ ]:
xy_df.head()


### Output metadata and coordinates into csvs

In [ ]:
metadata_out = Path("data/jdx_metadata.csv")
xy_out = Path("data/jdx_xy_points.csv")

metadata_out.parent.mkdir(parents=True, exist_ok=True)
metadata_df.to_csv(metadata_out, index=False)
xy_df.to_csv(xy_out, index=False)

print(f"Wrote {len(metadata_df)} metadata rows to {metadata_out}")
print(f"Wrote {len(xy_df)} xy rows to {xy_out}")

if not failed_df.empty:
    display(failed_df)


## Normalize compound id

In [ ]:

i = 0
_smiles_cache: dict = {}

try:
    with open("smiles_cache.json") as f:
        _smiles_cache = json.load(f)
except:
    pass

def save_cache():
    with open("smiles_cache.json", "w") as f:
        json.dump(_smiles_cache, f)

def as_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()

def normalize_cas(value: Any) -> str:
    text = as_text(value)
    if not text:
        return ""
    match = next((chunk for chunk in text.replace(";", " ").replace(",", " ").split() if chunk.count("-") == 2), "")
    return match.strip() if match else text.strip()

def is_valid_smiles(smiles: Any) -> bool:
    text = as_text(smiles)
    if not text:
        return False
    if Chem is None:
        return True
    return Chem.MolFromSmiles(text) is not None

def resolve_smiles_with_cirpy(identifier: str) -> str:
    if not identifier or cirpy is None:
        return ""
    try:
        result = cirpy.resolve(identifier, "smiles")
    except Exception:
        return ""
    return as_text(result)

def resolve_smiles_with_pubchem(identifier: str) -> str:
    if not identifier:
        return ""
    if identifier in _smiles_cache:
        return _smiles_cache[identifier]
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{quote(identifier, safe='')}/property/CanonicalSMILES/JSON"
    try:
        with urlopen(url, timeout=20) as r:
            data = json.loads(r.read())
            smiles = data["PropertyTable"]["Properties"][0]["CanonicalSMILES"]
            _smiles_cache[identifier] = smiles
            time.sleep(0.2)  # stay under PubChem's 5 req/sec limit
            return smiles
    except Exception:
        _smiles_cache[identifier] = ""
        return ""

def resolve_smiles_from_row(row: pd.Series) -> tuple[str, str, str]:
    global i
    i += 1
    if i % 100 == 0:
        save_cache()
        print(f"reached row {i}, cache size: {len(_smiles_cache)}")

    candidates = [
        ("cas registry no", normalize_cas(row.get("cas registry no"))),
        ("cas_registry_no", normalize_cas(row.get("cas_registry_no"))),
        ("title", as_text(row.get("title"))),
        ("names", as_text(row.get("names"))),
        ("name", as_text(row.get("name"))),
    ]

    for field, identifier in candidates:
        if not identifier:
            continue
        smiles = resolve_smiles_with_cirpy(identifier)
        if is_valid_smiles(smiles):
            return smiles, f"cirpy:{field}", identifier
        smiles = resolve_smiles_with_pubchem(identifier)
        if is_valid_smiles(smiles):
            return smiles, f"pubchem:{field}", identifier

    return "", "", ""

In [ ]:
smiles_values = metadata_df.apply(resolve_smiles_from_row, axis=1, result_type="expand")
smiles_values.columns = ["smiles", "smiles_method", "smiles_query"]
metadata_df = pd.concat([metadata_df, smiles_values], axis=1)
smiles_df = metadata_df.loc[metadata_df["smiles"].map(is_valid_smiles)].copy()


## Join spectroscopy metadata with xy points  

In [ ]:
smiles_df = pd.read_csv("/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR spectroscopy data/output.csv")
xy_df = pd.read_csv("/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR spectroscopy data/jdx_xy_points.csv")

In [ ]:
print("before join smiles df is ", smiles_df.shape)
print("before join xy df is ", xy_df.shape)

In [ ]:
# Perform the join operation
joined_df = pd.merge(smiles_df, xy_df, on="filename", how="inner")

print("Joined DataFrame head:")
display(joined_df.head())

In [ ]:
aggregated_xy_df = xy_df.groupby('filename').agg(x_coords=('x', list), y_coords=('y', list)).reset_index()

In [ ]:
metadata_with_aggregated_xy_df = pd.merge(smiles_df, aggregated_xy_df, on='filename', how='inner')

print("DataFrame with aggregated XY coordinates head:")
display(metadata_with_aggregated_xy_df.head())

In [ ]:
print(f"New DataFrame shape: {metadata_with_aggregated_xy_df.shape}")

In [ ]:
metadata_with_aggregated_xy_df.to_csv("data/IR spectroscopy data/NIST_IR_Spectroscopy.csv", index=False)
